### **Taller 3: Costrucción de un Modelo que prediga el precio de las casas en Boston empleando Algoritnos de 1. Arbol de Decisión, 2. Random Forest, 3.  Gradien Boosting, 4. Extreme Gradien Boosting, 5. Light Gradien Boosting**

#### Explicación del Notebook
**Problema:** Predecir precios de viviendas requiere capturar relaciones no lineales entre variables socioeconómicas y geográficas, comparando modelos para elegir el enfoque más robusto.

**Objetivo general:** Construir y comparar modelos de regresión basados en árboles para estimar el precio de casas y seleccionar el algoritmo con mejor desempeño.

**Objetivos específicos:**
1. Preparar el dataset y definir correctamente variables predictoras y variable objetivo.
2. Entrenar modelos de Árbol de Decisión, Random Forest, Gradient Boosting, XGBoost y LightGBM.
3. Evaluar y contrastar métricas de desempeño para identificar el modelo más preciso y estable.

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn statsmodels lightgbm

# 1 - Cargar Librerias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from sklearn import datasets, linear_model

# 2 - Cargar los Datos

In [ ]:
#Se usara un dataset disponible en gitbub
# https://raw.githubusercontent.com/selva86/datasets/refs/heads/master/BostonHousing.csv

# o la copia cargada en Github
# https://raw.githubusercontent.com/ivanacostag02/data-science-ml-portfolio/refs/heads/main/datasets/TALLER_03_Dataset_Boston_Housing.csv

In [ ]:
# En este caso usaremos la dataset dispoible en la ruta github
ruta_dataset = "https://raw.githubusercontent.com/ivanacostag02/data-science-ml-portfolio/refs/heads/main/datasets/TALLER_03_Dataset_Boston_Housing.csv"

In [ ]:
df_boston = pd.read_csv(ruta_dataset)

In [ ]:
type(df_boston)

In [ ]:
# en caso de que la dataset no este disponible, puede usar el siguiente respaldo en github
df_boston = pd.read_csv('https://raw.githubusercontent.com/ivanacostag02/data-science-ml-portfolio/refs/heads/main/datasets/TALLER_03_Dataset_Boston_Housing.csv')

In [ ]:
type(df_boston)

In [ ]:
df_boston.columns

In [ ]:
df_boston['PRICE'] = df_boston.medv

In [ ]:
df_boston.columns

In [ ]:
#boston2 = datasets.fetch_openml(name='boston', version=1, as_frame=True)
#print("Boston dataset loaded successfully.")

In [ ]:
#type(boston2)

In [ ]:
df_boston.shape

# 3 - Explorar los datos

In [ ]:
df_boston.info()

Para el Boston Housing clásico las columnas disponibles son:

**crim:** tasa de crimen per cápita por zona (crímenes por persona en el municipio).
​
**zn:** proporción de suelo residencial zonificado para lotes grandes (más de 25 000 pies²).
​
**indus:** proporción de acres destinados a uso industrial/comercial no minorista por municipio.
​
**chas:** indicador binario de cercanía al río Charles (1 = el tramo limita con el río, 0 = no).
​
**nox:** concentración de óxidos de nitrógeno en el aire (partes por 10 millones).
​
**rm:** número medio de habitaciones por vivienda.
​
**age:** porcentaje de viviendas ocupadas construidas antes de 1940 (viviendas “antiguas”).
​
**dis:** distancia media ponderada a cinco centros de empleo de Boston.
​
**rad:** índice de accesibilidad a autopistas radiales (valor categórico entero).
​
**tax:** tipo impositivo sobre la propiedad (tasa de impuesto a la propiedad por 10 000 dólares).
​
**ptratio:** ratio alumno/profesor por municipio.
​
**b:** variable derivada relacionada con la proporción de población negra (codifica diferencias raciales; tiene implicaciones éticas importantes).

**lstat:** porcentaje de población con estatus socioeconómico bajo.

**medv:** valor mediano de las viviendas ocupadas por sus dueños, en miles de dólares (es la variable objetivo original).

**PRICE:** en tu versión es otra columna numérica (float64) que guarda el mismo precio que medv pero renombrado (por ejemplo, para usarla como target en modelos). Es también una variable continua de tipo real.

# 4 - Dar tramiento a los datos crudos para obtener una Vista Minable

In [ ]:
# prompt: considera que se requiere realizar la limpieza de los datos del dataframe df_california, genera los comandos necesarios, evalua posibles datos perdiodos, presenta histograma de freecuencia y diagramas de cajas para detectar posible outliners

# 5 - Limpieza de datos
# Evaluar posibles datos perdidos
print("Missing values per column:")
print(df_boston.isnull().sum())

In [ ]:
# Histograma de frecuencia para cada columna para visualizar la distribución
df_boston.hist(figsize=(15, 10), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
# Diagramas de cajas para detectar posibles outliers
df_boston.plot(kind='box', subplots=True, layout=(4,4), figsize=(15,10), sharex=False, sharey=False)
plt.tight_layout()
plt.show()

In [ ]:
# prompt: genera el codigo para corregir los outliers superiores e inferiores

# Función para corregir outliers usando el método del rango intercuartílico (IQR)
def correct_outliers_iqr(df, column):
  Q1 = df[column].quantile(0.25)
  Q3 = df[column].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - (1.5 * IQR)
  upper_bound = Q3 + (1.5 * IQR)

  # Reemplazar outliers por los límites (opcional: podrías usar la mediana, etc.)
  df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
  df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
  return df

# Aplicar la corrección de outliers a las columnas numéricas (excluyendo PRICE ya que es el objetivo)
for col in df_boston.columns:
  if col != 'PRICE':
    df_boston = correct_outliers_iqr(df_boston, col)

# Mostrar diagramas de cajas después de la corrección para verificar
df_boston.plot(kind='box', subplots=True, layout=(4,4), figsize=(15,10), sharex=False, sharey=False)
plt.tight_layout()
plt.show()


# 5 - Divir los datos en entrada (Xs) y salida (Y)

In [ ]:
# prompt: separa a price como columna 'Y' y reserva el 70% para entrenmiento y 30% para pruneas

from sklearn.model_selection import train_test_split

X = df_boston.drop('PRICE', axis=1)
Y = df_boston['PRICE']


#**6 - Dividir los datos en Entrenamiento (train) en Pruebas (test)**

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of Y_train:", Y_train.shape)
print("Shape of Y_test:", Y_test.shape)

#**7 - Crear el Modelo**

Modelo basado en Algoritmo de Arbol de Decisión pararesolver problema de Regresión

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Árbol de Decisión con profundidad máxima de 5 (poda)
modelo_dt = DecisionTreeRegressor(max_depth=5)

#**8 - Entrenar el Modelo Arbol de Decisión (Decision TREE)**

In [ ]:

modelo_dt.fit(X_train, Y_train)



#**9 - Predicción del Modelo**

In [ ]:
# Predicción
Y_pred_dt = modelo_dt.predict(X_test)

#**10 - Validacion de Calidad del modelo**

In [ ]:
# Evaluación
mse_dt = mean_squared_error(Y_test, Y_pred_dt)
r2_dt = r2_score(Y_test, Y_pred_dt)

print(f'MSE Árbol de Decisión (poda a 5 niveles): {mse_dt}')
print(f'R² Árbol de Decisión (poda a 5 niveles): {r2_dt}')


In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20,15))
plot_tree(modelo_dt,
          feature_names=X.columns,
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Árbol de Decisión con Profundidad Máxima de 5')
plt.show()

#**11 - Arbol a 6 niveles de profundidad**

In [ ]:
# Árbol de Decisión con profundidad máxima de 6 (poda)
modelo_dt_v6n = DecisionTreeRegressor(max_depth=6)

In [ ]:
# Entrenamiento
modelo_dt_v6n.fit(X_train, Y_train)

In [ ]:
# Predicción
Y_pred_dt_v6n = modelo_dt_v6n.predict(X_test)

In [ ]:
# Evaluación
mse_dt_v6n = mean_squared_error(Y_test, Y_pred_dt_v6n)
r2_dt_v6n = r2_score(Y_test, Y_pred_dt_v6n)

print(f'MSE Árbol de Decisión (poda a 6 niveles): {mse_dt_v6n}')
print(f'R² Árbol de Decisión (poda a 6 niveles): {r2_dt_v6n}')

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20,15))
plot_tree(modelo_dt_v6n,
          feature_names=X.columns,
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Árbol de Decisión con Profundidad Máxima de 6')
plt.show()

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(40,30)) # Increased figure size significantly
plot_tree(modelo_dt_v6n,
          feature_names=X.columns,
          filled=True,
          rounded=True,
          fontsize=8, # Adjusted fontsize to fit in larger figure
          proportion=True # Proportionally adjust node size
         )
plt.title('Árbol de Decisión con Profundidad Máxima de 6 (Visualización Mejorada)', fontsize=20)
plt.show()

#**11 - Arbol a 7 niveles de profundidad**

In [ ]:
# Árbol de Decisión con profundidad máxima de 6 (poda)
modelo_dt_v7n = DecisionTreeRegressor(max_depth=7)

In [ ]:
# Entrenamiento
modelo_dt_v7n.fit(X_train, Y_train)

In [ ]:
# Predicción
Y_pred_dt_v7n = modelo_dt_v7n.predict(X_test)

In [ ]:
# Evaluación
mse_dt_v7n = mean_squared_error(Y_test, Y_pred_dt_v7n)
r2_dt_v7n = r2_score(Y_test, Y_pred_dt_v7n)

print(f'MSE Árbol de Decisión (poda a 7 niveles): {mse_dt_v7n}')
print(f'R² Árbol de Decisión (poda a 7 niveles): {r2_dt_v7n}')

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(40,30)) # Increased figure size significantly
plot_tree(modelo_dt_v7n,
          feature_names=X.columns,
          filled=True,
          rounded=True,
          fontsize=8, # Adjusted fontsize to fit in larger figure
          proportion=True # Proportionally adjust node size
         )
plt.title('Árbol de Decisión con Profundidad Máxima de 7 (Visualización Mejorada)', fontsize=20)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# 12 - Crear, Entrenar y Evaluar el Modelo Random Forest

# Crear el modelo Random Forest con 100 estimadores (bosquesitos)
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Entrenar el modelo
modelo_rf.fit(X_train, Y_train)

# Predicción
Y_pred_rf = modelo_rf.predict(X_test)

# Evaluación
mse_rf = mean_squared_error(Y_test, Y_pred_rf)
r2_rf = r2_score(Y_test, Y_pred_rf)

print(f'MSE Random Forest (100 estimadores): {mse_rf}')
print(f'R² Random Forest (100 estimadores): {r2_rf}')

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Crear el modelo Gradient Boosting
modelo_gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

# Entrenar el modelo
modelo_gb.fit(X_train, Y_train)

# Predicción
Y_pred_gb = modelo_gb.predict(X_test)

# Evaluación
mse_gb = mean_squared_error(Y_test, Y_pred_gb)
r2_gb = r2_score(Y_test, Y_pred_gb)

print(f'MSE Gradient Boosting: {mse_gb}')
print(f'R² Gradient Boosting: {r2_gb}')

In [ ]:
%pip install xgboost

import xgboost as xgb

# Crear el modelo XGBoost
modelo_xgb = xgb.XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=3, random_state=42)

# Entrenar el modelo
modelo_xgb.fit(X_train, Y_train)

# Predicción
Y_pred_xgb = modelo_xgb.predict(X_test)

# Evaluación
mse_xgb = mean_squared_error(Y_test, Y_pred_xgb)
r2_xgb = r2_score(Y_test, Y_pred_xgb)

print(f'MSE XGBoost: {mse_xgb}')
print(f'R² XGBoost: {r2_xgb}')

In [ ]:
# 15 - Modelo LightGBM
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score

# Crear el modelo LightGBM
modelo_lgb = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=42)

# Entrenar el modelo
modelo_lgb.fit(X_train, Y_train)

# Predicción
Y_pred_lgb = modelo_lgb.predict(X_test)

# Evaluación
mse_lgb = mean_squared_error(Y_test, Y_pred_lgb)
r2_lgb = r2_score(Y_test, Y_pred_lgb)

print(f'MSE LightGBM: {mse_lgb}')
print(f'R² LightGBM: {r2_lgb}')

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np


def registrar_metricas(nombre_modelo, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        "Modelo": nombre_modelo,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }


resultados = []

# Registrar solo si la prediccion existe (evita errores si falta ejecutar alguna celda)
if "Y_pred_dt" in globals():
    resultados.append(registrar_metricas("Decision Tree (max_depth=5)", Y_test, Y_pred_dt))

if "Y_pred_dt_v6n" in globals():
    resultados.append(registrar_metricas("Decision Tree (max_depth=6)", Y_test, Y_pred_dt_v6n))

if "Y_pred_dt_v7n" in globals():
    resultados.append(registrar_metricas("Decision Tree (max_depth=7)", Y_test, Y_pred_dt_v7n))

if "Y_pred_rf" in globals():
    resultados.append(registrar_metricas("Random Forest", Y_test, Y_pred_rf))

if "Y_pred_gb" in globals():
    resultados.append(registrar_metricas("Gradient Boosting", Y_test, Y_pred_gb))

if "Y_pred_xgb" in globals():
    resultados.append(registrar_metricas("XGBoost", Y_test, Y_pred_xgb))

if "Y_pred_lgb" in globals():
    resultados.append(registrar_metricas("LightGBM", Y_test, Y_pred_lgb))


df_resultados = pd.DataFrame(resultados)

if len(df_resultados) > 0:
    # Orden de mejor a peor: mayor R2 y menor RMSE
    df_resultados = (
        df_resultados.sort_values(by=["R2", "RMSE"], ascending=[False, True])
        .reset_index(drop=True)
    )

    print("Tabla comparativa de modelos:")
    display(df_resultados.round({"MSE": 4, "RMSE": 4, "MAE": 4, "R2": 4}))

    mejor = df_resultados.iloc[0]
    print(
        f"\nConclusion:\n"
        f"El modelo mas adecuado para estos datos es {mejor['Modelo']}, "
        f"porque obtuvo el mayor R2 ({mejor['R2']:.4f}) y un RMSE de {mejor['RMSE']:.4f}."
    )
else:
    print("No hay resultados para comparar. Ejecuta primero las celdas de entrenamiento y prediccion.")